# Column Transformers and Pipelines in Scikit-Learn

Using manual steps can lead to data leakage and verbose code. Scikit-Learn's `ColumnTransformer` and `Pipeline` allow us to bundle preprocessing steps and models together. This notebook covers:
*   `ColumnTransformer` to apply different prep steps to numeric vs. categorical columns.
*   `Pipeline` to chain simple imputation, scaling, and estimators.
*   Chaining preprocessing to a Decision Tree Regressor model.

In [35]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [36]:
df=pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")

In [37]:
df["brand"] = df["name"].str.split().str[0]

In [47]:
df.drop(columns='name',inplace=True)

In [48]:
df.describe()

,year,selling_price,km_driven
count,4340.000000,4.340000e+03,4340.000000
mean,2013.090783,5.041273e+05,66215.777419
std,4.215344,5.785487e+05,46644.102194
min,1992.000000,2.000000e+04,1.000000
25%,2011.000000,2.087498e+05,35000.000000
50%,2014.000000,3.500000e+05,60000.000000
75%,2016.000000,6.000000e+05,90000.000000
max,2020.000000,8.900000e+06,806599.000000


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   year           4340 non-null   int64 
 1   selling_price  4340 non-null   int64 
 2   km_driven      4340 non-null   int64 
 3   fuel           4340 non-null   object
 4   seller_type    4340 non-null   object
 5   transmission   4340 non-null   object
 6   owner          4340 non-null   object
 7   brand          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


In [50]:
cols=df.columns

In [51]:
df.isnull().sum()

year             0
selling_price    0
km_driven        0
fuel             0
seller_type      0
transmission     0
owner            0
brand            0
dtype: int64

In [52]:
print(cols)

Index(['year', 'selling_price', 'km_driven', 'fuel', 'seller_type',
       'transmission', 'owner', 'brand'],
      dtype='object')


In [53]:
for col in cols:
    print(col,df[col].nunique())

year 27
selling_price 445
km_driven 770
fuel 5
seller_type 3
transmission 2
owner 5
brand 29


In [55]:
x=df.drop(columns='selling_price')
y=df['selling_price']

In [56]:
x.head()

,year,km_driven,fuel,seller_type,transmission,owner,brand
0,2007,70000,Petrol,Individual,Manual,First Owner,Maruti
1,2007,50000,Petrol,Individual,Manual,First Owner,Maruti
2,2012,100000,Diesel,Individual,Manual,First Owner,Hyundai
3,2017,46000,Petrol,Individual,Manual,First Owner,Datsun
4,2014,141000,Diesel,Individual,Manual,Second Owner,Honda


In [81]:
df.shape

(4340, 8)

In [59]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [60]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2,random_state=42)

In [67]:
num_cols=xtrain.select_dtypes(include="number").columns
cat_cols=xtrain.select_dtypes(include='object').columns


In [91]:
xtrain.head()

,year,km_driven,fuel,seller_type,transmission,owner,brand
227,2017,20000,Diesel,Individual,Manual,First Owner,Mahindra
964,2018,50000,Diesel,Individual,Manual,First Owner,Maruti
2045,2013,25000,Petrol,Individual,Manual,Second Owner,Maruti
1025,2011,70000,Diesel,Individual,Manual,First Owner,Chevrolet
4242,2017,72000,Diesel,Dealer,Manual,First Owner,Maruti


In [92]:
cat_pipe=Pipeline([
    ("encoder",OneHotEncoder(handle_unknown='ignore'))
])
num_pipe=Pipeline([
    ("impute",SimpleImputer(strategy='median'))
])

In [93]:
preprocessor=ColumnTransformer(
    transformers=[
        ("num",num_pipe,num_cols),
        ("cat",cat_pipe,cat_cols)
    ]
)

In [94]:
model=DecisionTreeRegressor(
    criterion='squared_error',
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
    )

In [95]:
pipeline=Pipeline([
    ("preprocessor",preprocessor),
    ("model",model)
])

In [96]:
pipeline.fit(xtrain,ytrain)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [97]:
ypred=pipeline.predict(xtest)

In [98]:
xtest.shape

(868, 7)

In [99]:
ypred.shape

(868,)

In [100]:
import numpy as np
print("r2_score:",r2_score(ytest,ypred))
print("MAE:",mean_absolute_error(ytest,ypred))
print("RMSE:",np.sqrt(mean_squared_error(ytest,ypred)))

r2_score: 0.6591981049498365
MAE: 146399.3581454569
RMSE: 322494.3593469762


In [101]:
train_score = pipeline.score(xtrain, ytrain)
test_score = pipeline.score(xtest, ytest)

print("Train R² :", train_score)
print("Test R²  :", test_score)

Train R² : 0.8635390877330632
Test R²  : 0.6591981049498365
